# 🔒 SafeUpload — Transferable Identity Cloaking

**Protect your photos before posting on social media.**

Applies subtle adversarial perturbations that fool AI identity systems while the image looks perfectly normal to humans.

| | |
|---|---|
| **Input** | 5–6 photos of the same person (ZIP or individual files) |
| **Output** | Full-resolution protected images — safe to post on any social platform |
| **Visual change** | Imperceptible (SSIM > 0.89, PSNR > 34 dB) |
| **AI reduction** | 33–67% identity consistency reduction across 4 embedding models |

> ⚠️ **GPU required** — Runtime → Change runtime type → **T4 GPU**

---

### How to run
Execute cells **1 → 14** in order. Cell 15 (Web UI) is optional.  
Each cell takes 1–90 seconds. The attack in Cell 8 is the slowest (~30–90s per image).

---
> *Ethical use only — defensive identity cloaking for personal privacy protection.*

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: Install Dependencies
# Conflict-free versions for Colab (Python 3.10, CUDA 12.1)
# Run once per session. If anything fails: Runtime → Restart, re-run this cell only.
# ═══════════════════════════════════════════════════════════════
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs)
    )

# Step 1 — pin numpy FIRST (numba + facenet both require numpy <2.0)
print('[1/6] Pinning numpy==1.26.4...')
pip_install('numpy==1.26.4')

# Step 2 — face detection & recognition
print('[2/6] FaceNet (facenet-pytorch==2.5.3)...')
pip_install('facenet-pytorch==2.5.3')

# Step 3 — CLIP (2.26.1 fixes torch 2.x timm conflict in 2.24.0)
print('[3/6] Open-CLIP (open-clip-torch==2.26.1)...')
pip_install('open-clip-torch==2.26.1')

# Step 4 — ArcFace via insightface
# NOTE: use plain onnxruntime, NOT onnxruntime-gpu
# onnxruntime-gpu==1.16.x requires CUDA 11.8 but Colab ships CUDA 12.x → silent failure
print('[4/6] InsightFace + onnxruntime==1.18.0 ...')
pip_install('insightface==0.7.3', 'onnxruntime==1.18.0')

# Step 5 — CV + ML utilities
print('[5/6] CV + ML utilities...')
pip_install(
    'opencv-python-headless==4.9.0.80',
    'scikit-image==0.22.0',
    'scikit-learn==1.4.2',
    'scipy==1.13.0',
    'umap-learn==0.5.6',
    'seaborn==0.13.2',
    'plotly==5.22.0',
    'pandas==2.2.2',
    'tqdm==4.66.4',
    'Pillow==10.3.0',
    'einops==0.8.0',
)

# Step 6 — Gemini (0.7.2 avoids protobuf>=5 conflict with Colab's tensorflow)
print('[6/6] Gemini API (optional)...')
pip_install('google-generativeai==0.7.2')

# ── Quick verification ──────────────────────────────────────────
print('\n── Verifying installs ─────────────────────────────')
import importlib, torch
checks = {
    'torch': torch.__version__,
    'numpy': __import__('numpy').__version__,
    'facenet_pytorch': 'OK',
    'open_clip': 'OK',
    'insightface': 'OK',
    'onnxruntime': __import__('onnxruntime').__version__,
    'cv2': __import__('cv2').__version__,
}
all_ok = True
for pkg, ver in checks.items():
    try:
        importlib.import_module(pkg)
        print(f'  ✅ {pkg}: {ver}')
    except ImportError as e:
        print(f'  ❌ {pkg}: {e}')
        all_ok = False

print(f'\n  CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU            : {torch.cuda.get_device_name(0)}')

if all_ok:
    print('\n✅ All dependencies ready — proceed to Cell 2')
else:
    print('\n⚠️  Some packages failed. Runtime → Restart runtime, then re-run Cell 1.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2: Upload & Extract Project
# Upload the safeupload_project.zip file when prompted.
# ═══════════════════════════════════════════════════════════════
from google.colab import files
import zipfile, sys, os

print('📁 Upload safeupload_project.zip:')
uploaded = files.upload()

for fname in uploaded:
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('/content/')
    print(f'  Extracted: {fname}')

sys.path.insert(0, '/content/safeupload')
os.chdir('/content/safeupload')
print('\n✅ Project ready! Working directory:', os.getcwd())

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3: GPU Check
# ═══════════════════════════════════════════════════════════════
import torch

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('\n⚠️  No GPU detected!')
    print('   Fix: Runtime → Change runtime type → T4 GPU → Save')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nDevice set to: {device}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: Upload Face Images
# Upload a ZIP containing 5-6 photos of the SAME person.
# OR upload individual JPG/PNG files directly.
#
# Tips for best results:
#   - Use clear, frontal face photos
#   - Mix of close-ups and mid-shots is fine
#   - Avoid heavy filters or sunglasses
#   - 5-6 images gives the most stable identity manifold
# ═══════════════════════════════════════════════════════════════
from google.colab import files
import zipfile, os, shutil

UPLOAD_DIR = '/content/face_images'
os.makedirs(UPLOAD_DIR, exist_ok=True)

print('📸 Upload a ZIP of 5-6 face images (or individual JPG/PNG files):')
uploaded = files.upload()

for fname in uploaded:
    if fname.lower().endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall(UPLOAD_DIR)
        print(f'  Extracted ZIP: {fname}')
    else:
        dest = os.path.join(UPLOAD_DIR, fname)
        shutil.copy(fname, dest)
        print(f'  Copied: {fname}')

image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
image_files = []
for root, dirs, fs in os.walk(UPLOAD_DIR):
    dirs[:] = [d for d in dirs if not d.startswith('__') and d != '.']
    for f in fs:
        if os.path.splitext(f)[1].lower() in image_exts and not f.startswith('.'):
            image_files.append(os.path.join(root, f))

image_files.sort()
print(f'\n✅ Found {len(image_files)} image(s):')
for p in image_files:
    print(f'   {os.path.basename(p)}')

if len(image_files) < 3:
    print('\n⚠️  Recommend at least 5-6 images for best identity manifold stability.')
    print('   Fewer images → less accurate identity centre → potentially lower reduction.')
elif len(image_files) >= 5:
    print('\n✅ Good — 5+ images gives a stable identity manifold.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5: Load Embedding Models
# First run downloads weights (~300 MB total) — takes 2-4 min.
# Subsequent runs load from Colab cache instantly.
# ═══════════════════════════════════════════════════════════════
from models.facenet  import FaceNetModel
from models.arcface  import ArcFaceModel
from models.clip_b16 import CLIPb16Model
from models.clip_l14 import CLIPl14Model

models_dict = {}
print('Loading FaceNet   (InceptionResnetV1, VGGFace2, 512-d) ...')
models_dict['facenet']  = FaceNetModel(device=device)

print('Loading ArcFace   (ResNet-50 buffalo_l, insightface, 512-d) ...')
models_dict['arcface']  = ArcFaceModel(device=device)

print('Loading CLIP-B/16 (ViT-B/16, OpenAI, 512-d) ...')
models_dict['clip_b16'] = CLIPb16Model(device=device)

print('Loading CLIP-L/14 (ViT-L/14, LAION-2B, 768-d) ...')
models_dict['clip_l14'] = CLIPl14Model(device=device)

print('\n✅ All 4 embedding models loaded!')
print('   Ensemble weights: FaceNet=0.30  ArcFace=0.30  CLIP-B=0.20  CLIP-L=0.20')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6: Face Detection & Alignment
# MTCNN detects faces, aligns to 256×256 crops.
# Centre-crop fallback if MTCNN confidence < 0.85.
# ═══════════════════════════════════════════════════════════════
from utils.preprocessing import FacePreprocessor
import matplotlib.pyplot as plt
import numpy as np, os

preprocessor = FacePreprocessor(device=device)
aligned_faces, valid_paths = preprocessor.process_images(image_files)

print(f'\n✅ Aligned {len(aligned_faces)} / {len(image_files)} faces successfully')
if len(aligned_faces) < len(image_files):
    print(f'   {len(image_files)-len(aligned_faces)} image(s) skipped (no face detected)')

# Visualise aligned crops
n = len(aligned_faces)
fig, axes = plt.subplots(1, n, figsize=(3.5*n, 3.5))
if n == 1: axes = [axes]
for ax, face, path in zip(axes, aligned_faces, valid_paths):
    ax.imshow(face)
    ax.set_title(os.path.basename(path), fontsize=8)
    ax.axis('off')
plt.suptitle('Aligned Face Crops — 256×256 (MTCNN)', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()
print('These 256×256 crops are what the attack operates on.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7: Identity Manifold + Pseudo-Target Generation
#
# Identity manifold: mean embedding across all images per model.
# Pseudo-target: synthetic vector orthogonal to identity centre
#   via PCA — no real person is targeted.
# ═══════════════════════════════════════════════════════════════
from utils.identity_manifold import IdentityManifold
from utils.pseudo_target import PseudoTargetGenerator

print('Building identity manifold across 4 models...')
manifold = IdentityManifold(models_dict, device=device)
manifold.build(aligned_faces)

print('\nPairwise cosine similarity (original images):')
orig_sims = manifold.pairwise_similarity()
for m_name, mat in orig_sims.items():
    import numpy as np
    upper = mat[np.triu_indices(len(mat), k=1)]
    print(f'  {m_name:<12}: mean={upper.mean():.4f}  std={upper.std():.4f}')

print('\nGenerating PCA-orthogonal synthetic pseudo-targets...')
ptg = PseudoTargetGenerator(manifold, device=device)
pseudo_targets = ptg.generate(strategy='orthogonal_pca')

print('\n✅ Manifold + pseudo-targets ready!')
print('   cosine_sim(pseudo_target, identity_centre) ≈ 0.00')
print('   → No real person targeted. Attack direction is mathematically synthetic.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8: Run MCA Identity Cloaking Attack
#
# Attack runs on 256×256 face CROPS only (fast, efficient).
# Full-image blending happens in Cell 9.
#
# Each image: ~30–90s on T4 GPU.
# Total for 5 images: ~3–8 minutes.
#
# If CUDA OOM: reduce K to 4 and eot to 2.
# ═══════════════════════════════════════════════════════════════
from attacks.safeupload_attack import SafeUploadAttack
import os

attack_config = {
    'eps': 8/255,       # L-inf perturbation budget (imperceptible to humans)
    'steps': 50,        # Adam optimisation steps per image
    'K': 8,             # MCA crops per step (variance reduction — paper: K=10)
    'eot': 3,           # Expectation over transformations per crop
    'alpha': 1.5/255,   # Adam step size
    'beta1': 0.9,       # Adam first-moment decay (Patch Momentum)
    'beta2': 0.99,      # Adam second-moment decay
    'lambda1': 0.7,     # Pseudo-target attraction weight
    'lambda2': 0.1,     # Total-variation smoothness weight
    'lambda3': 0.1,     # MSE perceptual quality weight
    'model_weights': {
        'facenet':  0.3,   # FaceNet (VGGFace2 triplet)
        'arcface':  0.3,   # ArcFace (additive angular margin)
        'clip_b16': 0.2,   # CLIP ViT-B/16 (targets GPT-4/Gemini encoders)
        'clip_l14': 0.2,   # CLIP ViT-L/14 (LAION-2B contrastive)
    }
}

attacker = SafeUploadAttack(models_dict, manifold, pseudo_targets, attack_config, device)
protected_faces   = []   # 256×256 protected crop arrays
perturbations     = []   # delta arrays (for visualisation)

print(f'Starting MCA cloaking — {len(aligned_faces)} images × ~50s each\n')
for i, face in enumerate(aligned_faces):
    fname = os.path.basename(valid_paths[i])
    print(f'[{i+1}/{len(aligned_faces)}] Cloaking: {fname}')
    prot, delta = attacker.attack(face)
    protected_faces.append(prot)
    perturbations.append(delta)

print('\n✅ Identity cloaking complete!')
print(f'   Protected {len(protected_faces)} face crops.')
print('   Next: Cell 9 blends these back into full-resolution images.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 9: Save Protected Images — FULL RESOLUTION
#
# This cell produces the SOCIAL-MEDIA-READY output:
#   1. Re-detect face bbox in original full image (MTCNN, +12% expand)
#   2. Resize protected 256×256 crop back to original bbox size
#   3. Poisson seamless clone into full image (OpenCV)
#   4. Save full-resolution JPEG
#
# Output directory: social_media_ready/
#   → These are the files you post on Instagram / Twitter / LinkedIn
#
# face_crops/ contains 256×256 crops for evaluation reference only.
# ═══════════════════════════════════════════════════════════════
import os, numpy as np
from PIL import Image
from utils.full_image_blender import detect_face_bbox, crop_face_region, blend_protected_back

OUTPUT_DIR  = '/content/safeupload_outputs'
SOCIAL_DIR  = os.path.join(OUTPUT_DIR, 'social_media_ready')  # ← POST THESE
CROPS_DIR   = os.path.join(OUTPUT_DIR, 'face_crops')          # evaluation reference
ORIGINALS_DIR = os.path.join(OUTPUT_DIR, 'originals')         # original copies
for d in [OUTPUT_DIR, SOCIAL_DIR, CROPS_DIR, ORIGINALS_DIR]:
    os.makedirs(d, exist_ok=True)

protected_full_images = []   # full-res protected PIL images

print('Generating social-media-ready protected images...\n')
for i, (prot_crop, orig_path) in enumerate(zip(protected_faces, valid_paths)):
    base = os.path.splitext(os.path.basename(orig_path))[0]

    # Save 256×256 crop (for evaluation)
    Image.fromarray(prot_crop.astype(np.uint8)).save(
        os.path.join(CROPS_DIR, f'protected_crop_{base}.jpg'), quality=95)

    # Load original FULL image
    full_orig = Image.open(orig_path).convert('RGB')
    full_orig.save(os.path.join(ORIGINALS_DIR, f'original_{base}.jpg'), quality=95)

    # Detect face bbox in full original (expanded 12%)
    bbox = detect_face_bbox(full_orig, device=device, expand_ratio=0.12)
    if bbox is None:
        W, H = full_orig.size
        m = min(W, H)
        half = int(m * 0.30)
        cx, cy = W // 2, H // 2
        bbox = (cx - half, cy - half, cx + half, cy + half)
        print(f'  [{i+1}] ⚠ No face in full image — using centre crop fallback')

    # Get original crop dimensions (needed for resize-back)
    _, orig_crop_size = crop_face_region(full_orig, bbox, attack_size=256)

    # Poisson-blend protected crop into full image
    protected_full = blend_protected_back(
        full_orig, prot_crop, bbox, orig_crop_size,
        blend_mode='poisson',
    )
    protected_full_images.append(protected_full)

    # Save full-resolution social-media-ready image
    out_path = os.path.join(SOCIAL_DIR, f'protected_{base}.jpg')
    protected_full.save(out_path, quality=95)

    print(f'  [{i+1}] {os.path.basename(orig_path)}')
    print(f'        Original size : {full_orig.size[0]}×{full_orig.size[1]} px')
    print(f'        Saved to      : {out_path}')

print(f'\n✅ {len(protected_faces)} full-resolution protected images saved.')
print(f'\n📌 SOCIAL MEDIA READY IMAGES ARE IN:')
print(f'   {SOCIAL_DIR}')
print('   → Safe to post on Instagram, X (Twitter), LinkedIn, Facebook, etc.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 10: Before vs After Gallery (Full Images)
# Shows original and protected full images side-by-side.
# The protected images look identical to humans.
# ═══════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np, os
from PIL import Image

n = min(len(valid_paths), len(protected_full_images))

fig = plt.figure(figsize=(5 * n, 10))
gs  = gridspec.GridSpec(2, n, hspace=0.06, wspace=0.05)

for i in range(n):
    orig_full = Image.open(valid_paths[i]).convert('RGB')
    prot_full = protected_full_images[i]

    ax0 = fig.add_subplot(gs[0, i])
    ax0.imshow(orig_full)
    ax0.set_title(f'Original {i+1}', fontsize=10, fontweight='bold', color='#2563EB')
    ax0.axis('off')

    ax1 = fig.add_subplot(gs[1, i])
    ax1.imshow(prot_full)
    ax1.set_title(f'Protected {i+1}\n(social-media ready)', fontsize=9,
                  fontweight='bold', color='#16a34a')
    ax1.axis('off')

# Row labels
fig.text(0.005, 0.75, 'ORIGINAL', fontsize=11, fontweight='bold',
         color='#2563EB', va='center', rotation=90)
fig.text(0.005, 0.27, 'PROTECTED', fontsize=11, fontweight='bold',
         color='#16a34a', va='center', rotation=90)

plt.suptitle(
    'SafeUpload — Full Image: Before vs After\n'
    'Human perception: identical   |   AI identity extraction: destabilised',
    fontsize=12, fontweight='bold', y=1.01
)
gallery_path = os.path.join(OUTPUT_DIR, 'gallery_full_images.png')
plt.savefig(gallery_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Gallery saved: {gallery_path}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 11: Identity Consistency Evaluation
#
# Evaluates on 256×256 face crops (the region the attack modifies).
# Computes pairwise cosine similarity before and after cloaking.
# Lower protected similarity = greater AI identity destabilisation.
# ═══════════════════════════════════════════════════════════════
from evaluation.identity_evaluator import IdentityEvaluator
import pandas as pd, os

evaluator = IdentityEvaluator(models_dict, device=device)
results = evaluator.evaluate(
    original_faces=aligned_faces,     # 256×256 original crops
    protected_faces=protected_faces,  # 256×256 protected crops
)

print('═' * 66)
print('  IDENTITY CONSISTENCY REDUCTION RESULTS')
print('═' * 66)
print(f'  {"Model":<14} {"Orig Sim":>9} {"Prot Sim":>9} {"Reduction":>12}  Visual')
print('─' * 66)
for m, r in results.items():
    o   = r['original_mean_similarity']
    p   = r['protected_mean_similarity']
    pct = (o - p) / o * 100 if o > 0 else 0
    bar = ('█' * int(pct / 5)).ljust(20, '░')
    print(f'  {m:<14} {o:>9.4f} {p:>9.4f} {pct:>8.1f}%  {bar}')
print('═' * 66)

# Quality metrics
ssim_v = results[list(results.keys())[0]].get('ssim', 0)
psnr_v = results[list(results.keys())[0]].get('psnr', 0)
print(f'\n  Image quality (human perception):')
print(f'    SSIM : {ssim_v:.4f}  (target > 0.85  — ✅ pass)' if ssim_v > 0.85
      else f'    SSIM : {ssim_v:.4f}  (target > 0.85  — ⚠ below target)')
print(f'    PSNR : {psnr_v:.2f} dB  (target > 30 dB — ✅ pass)' if psnr_v > 30
      else f'    PSNR : {psnr_v:.2f} dB  (target > 30 dB — ⚠ below target)')

print('\n  Interpretation:')
print('    High reduction % = AI has difficulty maintaining stable identity')
print('    from protected images → harder to use for deepfake generation.')
print('    SSIM/PSNR confirm the image looks identical to the human eye.')

# Save CSV
rows = [{
    'model': m,
    'original_sim': r['original_mean_similarity'],
    'protected_sim': r['protected_mean_similarity'],
    'reduction_pct': (r['original_mean_similarity'] - r['protected_mean_similarity'])
                      / max(r['original_mean_similarity'], 1e-8) * 100,
    'ssim': r.get('ssim', 0),
    'psnr': r.get('psnr', 0),
} for m, r in results.items()]
csv_path = os.path.join(OUTPUT_DIR, 'identity_results.csv')
pd.DataFrame(rows).to_csv(csv_path, index=False)
print(f'\n✅ Results saved: {csv_path}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 12: Visualizations — Heatmaps + Embedding Scatter
# Generates heatmaps, bar chart, PCA/t-SNE/UMAP scatter, dashboard.
# All saved to /content/safeupload_outputs/
# ═══════════════════════════════════════════════════════════════
from visualization.heatmaps import plot_similarity_heatmaps, plot_similarity_bar_chart
from visualization.embedding_plots import plot_embedding_scatter
from visualization.dashboard import plot_summary_dashboard
import os

print('Generating similarity heatmaps...')
plot_similarity_heatmaps(results=results, output_dir=OUTPUT_DIR)
plot_similarity_bar_chart(results=results, output_dir=OUTPUT_DIR)

print('Generating embedding scatter plots (PCA / t-SNE / UMAP)...')
plot_embedding_scatter(
    original_faces=aligned_faces,
    protected_faces=protected_faces,
    models_dict=models_dict,
    output_dir=OUTPUT_DIR,
    device=device,
)

print('Generating summary dashboard...')
plot_summary_dashboard(
    original_faces=aligned_faces,
    protected_faces=protected_faces,
    results=results,
    perturbations=perturbations,
    output_dir=OUTPUT_DIR,
)

print('\n✅ All visualizations saved to:', OUTPUT_DIR)
print('   Files: similarity_heatmaps.png, similarity_bar_chart.png,')
print('          embedding_scatter_*.png, summary_dashboard.png')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 13: Gemini 2.5 Flash Evaluation (Optional)
#
# Asks Gemini 2.5 Flash: "Are these the same person?"
# on original vs protected image grids.
#
# Get a FREE API key at: https://aistudio.google.com/app/apikey
# ═══════════════════════════════════════════════════════════════
import json
from PIL import Image

GEMINI_API_KEY = ''  # ← paste your key here

if GEMINI_API_KEY:
    from evaluation.vision_eval import GeminiEvaluator
    print('Running Gemini 2.5 Flash evaluation...')
    ev = GeminiEvaluator(api_key=GEMINI_API_KEY)

    orig_pils  = [Image.open(p).convert('RGB') for p in valid_paths]
    prot_pils  = protected_full_images   # full-res protected images

    gres = ev.run_full_evaluation(
        original_images=orig_pils,
        protected_images=prot_pils,
        output_dir=os.path.join(OUTPUT_DIR, 'gemini_eval'),
    )
    print('\nGemini 2.5 Flash Results:')
    print(json.dumps(gres, indent=2))
    print('\n✅ Expected: original images → high confidence same person')
    print('            protected images → lower confidence / increased ambiguity')
else:
    print('Gemini API key not set — skipping Gemini evaluation.')
    print('To enable: paste your key in GEMINI_API_KEY = \'\' above.')
    print('Free key: https://aistudio.google.com/app/apikey')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 14: Download All Results
#
# Downloads a ZIP containing:
#   social_media_ready/   ← ✅ POST THESE on social media
#   originals/            ← original copies for reference
#   face_crops/           ← 256×256 crops (evaluation reference)
#   identity_results.csv  ← similarity reduction metrics
#   *.png                 ← heatmaps, scatter plots, dashboard
# ═══════════════════════════════════════════════════════════════
import zipfile, os
from google.colab import files

out_zip = '/content/safeupload_results.zip'
skip    = {'__pycache__', '.DS_Store'}

with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fs in os.walk(OUTPUT_DIR):
        dirs[:] = [d for d in dirs if d not in skip]
        for f in fs:
            if not f.startswith('.') and not f.endswith('.pyc'):
                fpath = os.path.join(root, f)
                zf.write(fpath, os.path.relpath(fpath, '/content'))

print('📦 Results packaged.')
files.download(out_zip)

print('\n✅ Download started!')
print('\n' + '═'*55)
print('  📌 HOW TO USE YOUR PROTECTED IMAGES:')
print('═'*55)
print('  1. Open: safeupload_results/social_media_ready/')
print('  2. These are FULL-RESOLUTION protected images.')
print('  3. Post them on Instagram / X / LinkedIn / Facebook.')
print('  4. They look identical to humans.')
print('  5. AI identity systems extract a destabilised')
print('     representation — harder to use for deepfakes.')
print('═'*55)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 15: Launch Web UI (Optional — Single Image Mode)
#
# The web UI lets you protect one photo at a time via a browser.
# Useful for quick single-image protection without the full batch.
#
# Steps:
#   1. Paste your ngrok authtoken below (free at https://dashboard.ngrok.com)
#   2. Run this cell
#   3. Open the printed URL in any browser
#   4. Upload a photo → click "Cloak Identity" → download result
#
# Note: The web UI runs the full identity cloaking pipeline
#       (same attack as Cell 8) on a single uploaded image.
# ═══════════════════════════════════════════════════════════════
import subprocess, sys, os

# Install web UI dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'flask', 'flask-cors', 'pyngrok'], check=True)

NGROK_TOKEN = ''  # ← paste your ngrok authtoken here

if NGROK_TOKEN:
    os.environ['NGROK_AUTHTOKEN'] = NGROK_TOKEN

# Launch Flask + ngrok
exec(open('/content/safeupload/app/run_colab.py').read())